# Algorithmic simplicity of topology-optimized metamaterials

Run top to bottom. Results are cached to `study.pkl`; delete it to recompute.

In [ ]:
import os, pickle, zlib, warnings, itertools
import numpy as np
import matplotlib; import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib.font_manager as fm
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve
from scipy.ndimage import label, find_objects, gaussian_filter
from scipy.stats import spearmanr, mannwhitneyu
warnings.filterwarnings("ignore")

N        = 32
NU       = 0.3
EMIN     = 1e-9
VF       = 0.4
RMIN     = 1.5
STAGES   = (1.0, 1.5, 2.0, 2.5, 3.0)
ITERS    = 120
TOL      = 1e-3
MOVE     = 0.15
AMP      = 0.9
MEASURES = ("clz", "zlib", "bdm")
THRESH   = 0.5

FULL      = True
N_SHEAR   = 2000 if FULL else 240
N_DIRX    = 500  if FULL else 160
N_RANDOM  = 2000 if FULL else 600
RMIN_SWEEP = (1.5, 2.0, 2.5, 3.0)
MESH_SWEEP = (16, 32, 64)
NU_SWEEP   = (-0.5, -0.25, 0.0, 0.2, 0.3, 0.45)
CACHE      = "study.pkl"
OUT        = "figures"; os.makedirs(OUT, exist_ok=True)

try:
    from joblib import Parallel, delayed
    JOBS = max(1, (os.cpu_count() or 2) - 1)
except ImportError:
    JOBS = 1

INK, OPT_C, RND_C, ACC_C, ALT_C = "#2b2b2b", "#2c5f7c", "#b3a99d", "#c17754", "#8a6d9e"
_serif = next((n for n in ["Times New Roman","Nimbus Roman","Liberation Serif","DejaVu Serif"]
               if any(f.name == n for f in fm.fontManager.ttflist)), "serif")
plt.rcParams.update({
    "font.family":"serif", "font.serif":[_serif], "mathtext.fontset":"stix",
    "axes.unicode_minus":False, "font.size":9, "axes.labelsize":8.4,
    "axes.labelcolor":INK, "text.color":INK, "xtick.color":INK, "ytick.color":INK,
    "xtick.labelsize":7.8, "ytick.labelsize":7.8, "axes.edgecolor":"#888888",
    "axes.linewidth":0.8, "axes.spines.top":False, "axes.spines.right":False,
    "legend.frameon":False, "savefig.bbox":"tight", "savefig.facecolor":"white",
    "svg.fonttype":"none", "savefig.dpi":300})
NICE = {"clz":"CLZ", "zlib":"DEFLATE", "bdm":"BDM"}
print(f"jobs={JOBS}  full={FULL}  serif={_serif}")

## Homogenization

In [ ]:
def _gauss(nu):
    D = 1.0/(1-nu**2)*np.array([[1,nu,0],[nu,1,0],[0,0,(1-nu)/2]])
    g = 1/np.sqrt(3); pts = [(-g,-g),(g,-g),(g,g),(-g,g)]
    XY = np.array([[0,0],[1,0],[1,1],[0,1]], float)
    B, w = [], []
    for xi, et in pts:
        dN = 0.25*np.array([[-(1-et),(1-et),(1+et),-(1+et)],
                            [-(1-xi),-(1+xi),(1+xi),(1-xi)]])
        J = dN @ XY; dNx = np.linalg.solve(J, dN)
        b = np.zeros((3,8))
        b[0,0::2]=dNx[0]; b[1,1::2]=dNx[1]; b[2,0::2]=dNx[1]; b[2,1::2]=dNx[0]
        B.append(b); w.append(abs(np.linalg.det(J)))
    return D, np.array(B), np.array(w)

class Cell:
    """Periodic unit cell, plane stress, SIMP interpolation."""
    def __init__(self, n, nu=NU, Emin=EMIN, p=3.0):
        self.n, self.Emin, self.p = n, Emin, p
        self.D, self.B, self.w = _gauss(nu)
        ix, iy = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
        nod = lambda a, b: ((a % n)*n + (b % n)).ravel()
        cs = [nod(ix,iy), nod(ix+1,iy), nod(ix+1,iy+1), nod(ix,iy+1)]
        ed = np.zeros((n*n,8), np.int64)
        for k, c in enumerate(cs): ed[:,2*k] = 2*c; ed[:,2*k+1] = 2*c+1
        self.ed, self.ne, self.ndof = ed, n*n, 2*n*n
        self.ke = sum(self.w[q]*self.B[q].T@self.D@self.B[q] for q in range(4))
        self.f0 = sum(self.w[q]*self.B[q].T@self.D for q in range(4))
        self.iK = np.repeat(ed, 8, axis=1).ravel()
        self.jK = np.tile(ed, (1,8)).ravel()
        self.free = np.arange(2, self.ndof)
        self.eps0 = np.eye(3)

    def _E(self, x):
        return self.Emin + np.clip(x.ravel(),0,1)**self.p*(1.0-self.Emin)

    def solve(self, x):
        E = self._E(x)
        K = coo_matrix(((self.ke.ravel()[None,:]*E[:,None]).ravel(), (self.iK,self.jK)),
                       shape=(self.ndof,self.ndof)).tocsr()
        F = np.zeros((self.ndof,3))
        for a in range(3):
            np.add.at(F[:,a], self.ed.ravel(), (self.f0[:,a][None,:]*E[:,None]).ravel())
        chi = np.zeros((self.ndof,3))
        chi[self.free] = spsolve(K[self.free][:,self.free].tocsc(), F[self.free])
        return chi, E

    def CH(self, x, sens=False):
        chi, E = self.solve(x)
        ue = chi[self.ed]
        ce = np.zeros((self.ne,3,3))
        for q in range(4):
            s = np.einsum("ij,eja->eia", self.B[q], ue)
            d = self.eps0[None,:,:] - s
            ce += self.w[q]*np.einsum("eia,ij,ejb->eab", d, self.D, d)
        C = (E[:,None,None]*ce).sum(0)/self.ne
        if not sens: return C
        dE = self.p*np.clip(x.ravel(),0,1)**(self.p-1)*(1.0-self.Emin)
        return C, dE[:,None,None]*ce/self.ne

D0, _, _ = _gauss(NU)
assert np.abs(Cell(8).CH(np.ones(64)) - D0).max() < 1e-12
print("solid cell reproduces the plane-stress matrix")

## Optimizer

In [ ]:
def build_filter(n, rmin):
    r = int(np.ceil(rmin)-1); I, J, V = [], [], []
    for i in range(n):
        for j in range(n):
            for di in range(-r, r+1):
                for dj in range(-r, r+1):
                    w = rmin - np.hypot(di, dj)
                    if w > 0:
                        I.append(i*n+j); J.append(((i+di)%n)*n+((j+dj)%n)); V.append(w)
    H = coo_matrix((V,(I,J)), shape=(n*n,n*n)).tocsr()
    return H, np.asarray(H.sum(1)).ravel()

def optimize(obj="shear", seed=0, n=N, nu=NU, vf=VF, rmin=RMIN, emin=EMIN,
             stages=STAGES, iters=ITERS, tol=TOL, move=MOVE, amp=AMP):
    comp = 2 if obj == "shear" else 0
    rng = np.random.default_rng(seed)
    cell = Cell(n, nu=nu, Emin=emin)
    H, Hs = build_filter(n, rmin)
    x = np.clip(vf + amp*(rng.random(n*n)-0.5), 1e-3, 1.0); x *= vf/x.mean()
    conv = False
    for si, p in enumerate(stages):
        cell.p = p; last = si == len(stages)-1
        for it in range(iters):
            xp = np.asarray(H @ x).ravel()/Hs
            C, dC = cell.CH(xp, sens=True)
            g = np.asarray(H.T @ (dC[:,comp,comp]/Hs)).ravel()
            lo, hi = 1e-12, 1e9
            while (hi-lo)/(hi+lo) > 1e-6:
                mid = 0.5*(lo+hi)
                xn = np.clip(np.clip(x*np.sqrt(np.maximum(g,1e-30)/mid),
                                     x-move, x+move), 0, 1)
                if (np.asarray(H @ xn).ravel()/Hs).mean() > vf: lo = mid
                else: hi = mid
            ch = np.abs(xn-x).max(); x = xn
            if ch < tol and it >= 20:
                conv = last; break
        else:
            conv = False
    xf = (np.asarray(H @ x).ravel()/Hs).reshape(n, n)
    xb = (xf > THRESH).astype(np.uint8)
    return dict(xf=xf, xb=xb, C=Cell(n, nu=nu, Emin=emin).CH(xb.astype(float).ravel()),
                conv=bool(conv), obj=obj, seed=seed)

## Complexity estimators

In [ ]:
def lz76(s):
    n = len(s); i, k, l, c, kmax = 0, 1, 1, 1, 1
    while True:
        if s[i+k-1] == s[l+k-1]:
            k += 1
            if l+k > n: c += 1; break
        else:
            if k > kmax: kmax = k
            i += 1
            if i == l:
                c += 1; l += kmax
                if l+1 > n: break
                i = 0; k = 1; kmax = 1
            else: k = 1
    return c

try:
    from pybdm import BDM
    _bdm = BDM(ndim=2)
    def _bdm_of(a): return float(_bdm.bdm(np.asarray(a, int)))
except Exception:
    _bdm = None
    def _bdm_of(a):
        a = np.asarray(a, np.uint8); b = []
        for i in range(0, a.shape[0], 4):
            for j in range(0, a.shape[1], 4):
                b.append(a[i:i+4, j:j+4].tobytes())
        u, c = np.unique(b, return_counts=True)
        return float(np.sum(np.log2(c)+1) + len(u))

def K(xb, measure="zlib"):
    a = np.asarray(xb, np.uint8)
    if measure == "zlib": return float(len(zlib.compress(a.tobytes(), 9)))
    if measure == "clz":  return float(lz76("".join(map(str, a.ravel()))))
    if measure == "bdm":  return _bdm_of(a)
    raise ValueError(measure)

assert lz76("0"*64) < lz76("".join(map(str, np.random.default_rng(0).integers(0,2,64))))
print("estimators ready" + ("" if _bdm else "  (pybdm absent, block fallback in use)"))

## Identity and baselines

In [ ]:
def d4(a):
    out, c = [], np.asarray(a, np.uint8)
    for _ in range(4):
        out += [np.ascontiguousarray(c), np.ascontiguousarray(np.fliplr(c))]
        c = np.rot90(c)
    return out

def key_d4(a):
    return min(t.tobytes() for t in d4(a))

def key_full(a):
    a = np.asarray(a, np.uint8); n = a.shape[0]; best = None
    for dy in range(n):
        r = np.roll(a, dy, 0)
        for dx in range(n):
            k = key_d4(np.roll(r, dx, 1))
            if best is None or k < best: best = k
    return best

def distinct(cells, key=key_full):
    seen, out = set(), []
    for c in cells:
        h = key(c["xb"])
        if h not in seen: seen.add(h); out.append(c)
    return out

def random_cell(rng, n=N, vf=VF, rmin=RMIN, sigma=1.0):
    f = gaussian_filter(rng.normal(size=(n,n)), sigma, mode="wrap")
    x = (f >= np.quantile(f, 1-vf)).astype(float)
    H, Hs = _FILT.setdefault((n,rmin), build_filter(n,rmin))
    xf = (np.asarray(H @ x.ravel()).ravel()/Hs).reshape(n,n)
    return (xf > THRESH).astype(np.uint8)

def surrogate(xf, rng):
    phi = float((np.asarray(xf) >= THRESH).mean())
    F = np.fft.fft2(np.asarray(xf, float))
    ph = rng.uniform(0, 2*np.pi, F.shape); ph[0,0] = 0.0
    g = np.real(np.fft.ifft2(np.abs(F)*np.exp(1j*ph)))
    return (g >= np.quantile(g, 1-phi)).astype(np.uint8)

def percolates(xb):
    a = np.asarray(xb, np.uint8); n = a.shape[0]
    lab, m = label(np.tile(a, (3,3)))
    if m == 0: return False
    for sl in find_objects(lab):
        if sl is None: continue
        if (sl[0].stop-sl[0].start) > n or (sl[1].stop-sl[1].start) > n: return True
    return False

def gray_fraction(xf):
    x = np.asarray(xf)
    return float(((x > 0.1) & (x < 0.9)).mean())

_FILT = {}
print("ready")

## Run

In [ ]:
def _one(obj, seed, **kw): return optimize(obj=obj, seed=seed, **kw)

def run_many(obj, nruns, **kw):
    if JOBS > 1:
        return Parallel(n_jobs=JOBS, verbose=0)(
            delayed(_one)(obj, s, **kw) for s in range(nruns))
    return [_one(obj, s, **kw) for s in range(nruns)]

if os.path.exists(CACHE):
    S = pickle.load(open(CACHE, "rb")); print(f"loaded {CACHE}")
else:
    S = {"designs": {}, "sweeps": {}}
    print("shear ...");       S["designs"]["shear"] = run_many("shear", N_SHEAR)
    print("directional ..."); S["designs"]["dirx"]  = run_many("dirx",  N_DIRX)

    rng = np.random.default_rng(12345)
    S["random"] = [dict(xb=random_cell(rng)) for _ in range(N_RANDOM)]
    for c in S["random"]:
        c["C"] = Cell(N).CH(c["xb"].astype(float).ravel())

    rng = np.random.default_rng(999)
    opt = [d for d in S["designs"]["shear"] if d["conv"]] or S["designs"]["shear"]
    S["matched"] = [dict(xb=surrogate(d["xf"], rng)) for d in opt[:200]]

    for r in RMIN_SWEEP[1:]:
        S["sweeps"][f"rmin={r}"] = dict(
            shear=run_many("shear", max(40, N_SHEAR//8), rmin=r),
            dirx=run_many("dirx",  max(40, N_DIRX//4),  rmin=r))
    for n in MESH_SWEEP:
        if n == N: continue
        S["sweeps"][f"N={n}"] = dict(
            shear=run_many("shear", max(30, N_SHEAR//12), n=n, rmin=1.5*n/32))
    for v in NU_SWEEP:
        if v == NU: continue
        S["sweeps"][f"nu={v}"] = dict(shear=run_many("shear", max(30, N_SHEAR//12), nu=v))
    pickle.dump(S, open(CACHE, "wb")); print(f"saved {CACHE}")

for o in ("shear","dirx"):
    d = S["designs"][o]; cv = [x for x in d if x["conv"]]
    print(f"{o:>12}: {len(d)} runs, {len(cv)} converged, "
          f"{len(distinct(cv or d, key_d4))} distinct (D4), "
          f"{len(distinct(cv or d))} distinct (D4+translation)")

## Complexity and effect sizes

In [ ]:
def cliffs_delta(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    gt = (a[:,None] > b[None,:]).sum(); lt = (a[:,None] < b[None,:]).sum()
    return (gt-lt)/(len(a)*len(b))

def complexities(cells, measures=MEASURES):
    return {m: np.array([K(c["xb"], m) for c in cells]) for m in measures}

OPT = {o: distinct([d for d in S["designs"][o] if d["conv"]] or S["designs"][o])
       for o in ("shear","dirx")}
RND  = S["random"]; MAT = S["matched"]
KO  = {o: complexities(OPT[o]) for o in OPT}
KR  = complexities(RND)
KM  = complexities(MAT)

rows = []
for o in ("shear","dirx"):
    for m in MEASURES:
        d  = cliffs_delta(KO[o][m], KR[m])
        dm = cliffs_delta(KO[o][m], KM[m])
        p  = mannwhitneyu(KO[o][m], KR[m], alternative="less").pvalue
        rows.append((o, m, len(KO[o][m]), len(KR[m]), d, dm, p))
print(f"{'obj':>6} {'est':>8} {'M':>5} {'n':>6} {'delta':>8} {'delta_m':>8} {'p':>10}")
for o,m,M,n,d,dm,p in rows:
    print(f"{o:>6} {NICE[m]:>8} {M:>5} {n:>6} {d:>8.2f} {dm:>8.2f} {p:>10.2e}")

## Figure 1

In [ ]:
fig = plt.figure(figsize=(7.2,2.6))
gs = gridspec.GridSpec(1,3, wspace=.34, left=.07, right=.985, top=.88, bottom=.19)
for k,(o,comp,nm) in enumerate((("shear",2,"shear"),("dirx",0,"directional"))):
    ax = fig.add_subplot(gs[0,k])
    ax.set_title(f"({'ab'[k]})  {nm}", loc="left", fontsize=8.8, pad=6)
    ax.scatter(KR["zlib"], [max(c["C"][comp,comp],1e-12) for c in RND],
               s=9, c=RND_C, edgecolor="none", label="random")
    ax.scatter(KO[o]["zlib"], [max(c["C"][comp,comp],1e-12) for c in OPT[o]],
               s=14, c=OPT_C, edgecolor="none", label="optima")
    ax.set_yscale("log"); ax.set_xlabel("complexity (DEFLATE, bytes)")
    ax.set_ylabel(f"$C_{{{comp+1}{comp+1}}}$")
    ax.legend(fontsize=7, loc="upper right", frameon=True, facecolor="white",
              edgecolor="none", framealpha=.9, handletextpad=.3, borderpad=.3)
ax = fig.add_subplot(gs[0,2])
ax.set_title("(c)  effect size", loc="left", fontsize=8.8, pad=6)
w = .35
for i,o in enumerate(("shear","dirx")):
    v = [cliffs_delta(KO[o][m], KR[m]) for m in MEASURES]
    ax.bar(np.arange(len(MEASURES))+ (i-.5)*w, v, w,
           color=[OPT_C,ACC_C][i], label=["shear","directional"][i])
ax.set_xticks(range(len(MEASURES))); ax.set_xticklabels([NICE[m] for m in MEASURES])
ax.set_ylabel("Cliff's $\\delta$"); ax.set_ylim(-1.1, 0.34)
ax.axhline(0, color="#999999", lw=.8)
ax.legend(fontsize=7, loc="upper right", handletextpad=.3, borderpad=.2)
fig.savefig(f"{OUT}/fig1.svg"); fig.savefig(f"{OUT}/fig1.png"); plt.show()

## Figure 2

In [ ]:
src = (OPT["shear"] or OPT["dirx"])[0]
xf  = src["xf"]; phi = float((xf >= THRESH).mean())
F   = np.fft.fft2(xf); amp = np.fft.fftshift(np.abs(F))
rng2 = np.random.default_rng(7)
th = rng2.uniform(0, 2*np.pi, F.shape); th[0,0] = 0.0
gfield = np.real(np.fft.ifft2(np.abs(F)*np.exp(1j*th)))
sur = (gfield >= np.quantile(gfield, 1-phi)).astype(np.uint8)

def psd_radial(xb):
    x = np.asarray(xb, float); n = x.shape[0]
    P = np.fft.fftshift(np.abs(np.fft.fft2(x-x.mean()))**2)/x.size
    c = n//2; yy, xx = np.mgrid[0:n,0:n]; dd = np.hypot(yy-c, xx-c)
    return (np.arange(1,c+1,dtype=float),
            np.array([P[(dd>=i-.5)&(dd<i+.5)].mean() for i in range(1,c+1)]))

def s2_radial(xb):
    x = np.asarray(xb, float); n = x.shape[0]
    F2 = np.fft.fft2(x)
    Sp = np.fft.fftshift(np.real(np.fft.ifft2(F2*np.conj(F2)))/x.size)
    c = n//2; yy, xx = np.mgrid[0:n,0:n]; dd = np.hypot(yy-c, xx-c)
    return (np.arange(c+1,dtype=float),
            np.array([Sp[(dd>=i-.5)&(dd<i+.5)].mean() for i in range(c+1)]))

fig = plt.figure(figsize=(7.2,4.4))
gs = gridspec.GridSpec(2,12, height_ratios=[1,1.35], hspace=.70, wspace=1.9,
                       left=.07, right=.985, top=.895, bottom=.105)
steps = [(gs[0,0:3],  xf,                 r"optimum  $x_f$",            True,  "gray"),
         (gs[0,3:6],  np.log10(amp+1e-9), r"$|\hat{I}(\mathbf{k})|$",   False, "magma"),
         (gs[0,6:9],  gfield,             "phase randomized",           False, "gray"),
         (gs[0,9:12], sur,                "surrogate",                  True,  "gray")]
axs = []
for cell_, arr, ttl, inv, cm in steps:
    ax = fig.add_subplot(cell_)
    ax.imshow(1-arr if inv else arr, cmap=cm)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(ttl, fontsize=7.6, pad=4)
    axs.append(ax)
fig.text(.012,.95,"(a)", fontsize=10, fontweight="bold")
labels = [r"$\mathcal{F}$",
          "$|\\hat{I}|e^{i\\theta}$\n$\\theta\\sim U(0,2\\pi)$",
          "threshold\nat $\\phi$"]
for k, lab in enumerate(labels):
    p0, p1 = axs[k].get_position(), axs[k+1].get_position()
    y = (p0.y0+p0.y1)/2
    fig.patches.append(plt.matplotlib.patches.FancyArrowPatch(
        (p0.x1+.005, y), (p1.x0-.005, y), transform=fig.transFigure,
        color=INK, arrowstyle="-|>", mutation_scale=8, lw=.9))
    fig.text((p0.x1+p1.x0)/2, y+.045, lab, fontsize=6.3, ha="center",
             va="bottom", color=INK, linespacing=1.3)

pops = {"optima": [c["xb"] for c in OPT["shear"]+OPT["dirx"]][:120],
        "filtered random": [c["xb"] for c in RND[:120]],
        "correlation matched": [c["xb"] for c in MAT[:120]]}
axP = fig.add_subplot(gs[1,0:4]); axS = fig.add_subplot(gs[1,4:8])
axD = fig.add_subplot(gs[1,8:12])
for (nm, cells), c, ls in zip(pops.items(), (OPT_C, RND_C, ACC_C), ("-","--",":")):
    kk = psd_radial(cells[0])[0]
    Pm = np.vstack([psd_radial(z)[1] for z in cells]).mean(0); Pm = Pm/Pm.sum()
    rr = s2_radial(cells[0])[0]
    Sm = np.vstack([s2_radial(z)[1] for z in cells]).mean(0)
    axP.plot(kk, Pm, ls, color=c, lw=1.4, label=nm)
    axS.plot(rr, Sm, ls, color=c, lw=1.4)
axP.set_xscale("log"); axP.set_yscale("log")
axP.set_title(r"(b)  $P(k)=|\hat{I}(\mathbf{k})|^{2}/N^{2}$", loc="left",
              fontsize=8.4, pad=6)
axP.set_xlabel(r"wavenumber $k$"); axP.set_ylabel("normalized power")
axP.legend(fontsize=6.2, loc="lower left")
axS.set_title(r"(c)  $S_2(r)=\langle I(\mathbf{x})\,I(\mathbf{x}+\mathbf{r})\rangle$",
              loc="left", fontsize=8.4, pad=6)
axS.set_xlabel(r"separation $r$"); axS.set_ylabel(r"$S_2(r)$")
axS.text(.97,.93, "$S_2(0)=\\phi$\n$S_2(\\infty)\\to\\phi^{2}$", transform=axS.transAxes,
         fontsize=6.6, ha="right", va="top", color="#5a5a5a", linespacing=1.4)
w = .35
for i, (lab, Kb) in enumerate((("filtered", KR), ("matched", KM))):
    axD.bar(np.arange(len(MEASURES))+(i-.5)*w,
            [cliffs_delta(KO["shear"][m], Kb[m]) for m in MEASURES], w,
            color=[RND_C, ACC_C][i], label=lab)
axD.set_title(r"(d)  Cliff's $\delta$ against each baseline", loc="left",
              fontsize=8.4, pad=6)
axD.set_xticks(range(len(MEASURES))); axD.set_xticklabels([NICE[m] for m in MEASURES])
axD.set_ylim(-1.1,.34); axD.axhline(0, color="#999999", lw=.8)
axD.legend(fontsize=6.6, loc="upper right")
fig.savefig(f"{OUT}/fig2.svg"); fig.savefig(f"{OUT}/fig2.png"); plt.show()

## Figure 3

In [ ]:
gf = np.array([gray_fraction(c["xf"]) for c in OPT["shear"]])
lo = gf <= np.median(gf)
ths = np.linspace(0.3, 0.7, 9)
sweep = {m: [] for m in MEASURES}
for t in ths:
    ko = {m: np.array([K((c["xf"]>t).astype(np.uint8), m) for c in OPT["shear"]])
          for m in MEASURES}
    kr = {m: np.array([K(c["xb"], m) for c in RND[:300]]) for m in MEASURES}
    for m in MEASURES: sweep[m].append(cliffs_delta(ko[m], kr[m]))

fig = plt.figure(figsize=(7.2,2.6))
gs = gridspec.GridSpec(1,2, wspace=.30, left=.08, right=.985, top=.88, bottom=.19)
ax = fig.add_subplot(gs[0,0]); ax.set_title("(a)  gray fraction", loc="left",
                                            fontsize=8.8, pad=6)
ax.scatter(gf, KO["shear"]["zlib"], s=14, c=OPT_C, edgecolor="none")
ax.set_xlabel("gray fraction of the filtered field"); ax.set_ylabel("complexity")
ax.text(.03,.05,f"least-gray half: $\\delta$ = "
        f"{cliffs_delta(KO['shear']['zlib'][lo], KR['zlib']):+.2f}",
        transform=ax.transAxes, fontsize=7.6)
ax = fig.add_subplot(gs[0,1]); ax.set_title("(b)  binarization threshold", loc="left",
                                            fontsize=8.8, pad=6)
for m,c in zip(MEASURES,(OPT_C,ACC_C,ALT_C)):
    ax.plot(ths, sweep[m], "o-", ms=3.4, lw=1.2, color=c, label=NICE[m])
ax.axvline(THRESH, color="#999999", lw=.8, ls="--")
ax.set_xlabel("threshold"); ax.set_ylabel("Cliff's $\\delta$")
ax.set_ylim(-1.08,.08); ax.legend(fontsize=7, loc="lower right")
fig.savefig(f"{OUT}/fig3.svg"); fig.savefig(f"{OUT}/fig3.png"); plt.show()

## Figure 4

In [ ]:
def tkey(Cm, tol=0.01):
    v = np.array([Cm[0,0], Cm[1,1], Cm[0,1], Cm[2,2]], float)
    return tuple((v/tol).round().astype(int))

pool = OPT["shear"] + OPT["dirx"] + RND
Kp   = np.array([K(c["xb"], "zlib") for c in pool])
pc   = np.array([percolates(c["xb"]) for c in pool])
grp  = {}
for i,c in enumerate(pool): grp.setdefault(tkey(c["C"]), []).append(i)
sz   = np.array([len(v) for v in grp.values()], float)
base = float(np.median(Kp[~pc])) if (~pc).any() else float(np.median(Kp))

fig = plt.figure(figsize=(7.2,2.6))
gs = gridspec.GridSpec(1,2, width_ratios=[1,.8], wspace=.30,
                       left=.08, right=.985, top=.88, bottom=.19)
ax = fig.add_subplot(gs[0,0]); ax.set_title("(a)  neutral-set sizes", loc="left",
                                            fontsize=8.8, pad=6)
ax.hist(sz, bins=np.logspace(0, np.log10(max(sz.max(),2)), 26), color=RND_C)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("topologies sharing one tensor"); ax.set_ylabel("number of tensors")
ax = fig.add_subplot(gs[0,1]); ax.set_title("(b)  at each optimum", loc="left",
                                            fontsize=8.8, pad=6)
vals = [len(grp.get(tkey(OPT[o][0]["C"]), [])) for o in ("dirx","shear")]
ax.bar(range(2), [max(v,.5) for v in vals], .5, color=[ACC_C,OPT_C])
ax.set_yscale("log")
for i,v in enumerate(vals): ax.text(i, max(v,.5)*1.3, str(v), ha="center", fontsize=8)
ax.set_xticks(range(2)); ax.set_xticklabels(["directional","shear"])
ax.set_ylabel("neutral-set size at optimum"); ax.set_ylim(.4, max(vals)*4)
fig.savefig(f"{OUT}/fig4.svg"); fig.savefig(f"{OUT}/fig4.png"); plt.show()

## Figure 5

In [ ]:
runs = [d for d in S["designs"]["dirx"] if d["conv"]] or S["designs"]["dirx"]
cnt, rep = {}, {}
for d in runs:
    h = key_full(d["xb"]); cnt[h] = cnt.get(h,0)+1; rep.setdefault(h, d)
ks = list(cnt); b = np.array([cnt[h] for h in ks], float)
kk = np.array([K(rep[h]["xb"], "zlib") for h in ks]); kk = kk/np.median(kk)
one, many = b < 2, b >= 2
mk = {m: np.array([K(rep[h]["xb"], m) for h in ks]) for m in MEASURES}
pv = {m: mannwhitneyu(mk[m][many], mk[m][one], alternative="less").pvalue
      if many.any() and one.any() else np.nan for m in MEASURES}
mdK = np.array([np.median(Kp[np.array(v)]) for v in grp.values()])
rho = spearmanr(sz, mdK)[0]

fig = plt.figure(figsize=(7.2,3.0))
gs = gridspec.GridSpec(1,3, width_ratios=[1,.86,.26], wspace=.26,
                       left=.08, right=.985, top=.855, bottom=.185)
ax = fig.add_subplot(gs[0,0])
ax.set_title("(a)  degeneracy does not explain it", loc="left", fontsize=8.8, pad=7)
ax.scatter(sz, mdK, s=14, c=RND_C, edgecolor="none")
ax.axhline(base, color=ALT_C, lw=1.1, ls=(0,(5,3)))
ax.set_xscale("log"); ax.set_xlabel("topologies sharing one tensor")
ax.set_ylabel("median complexity")
ax.text(.03,.05, f"$\\rho$ = {rho:+.2f}", transform=ax.transAxes, fontsize=9)
ax = fig.add_subplot(gs[0,1])
ax.set_title("(b)  the optimizer does", loc="left", fontsize=8.8, pad=7)
ax.scatter(b[one], kk[one], s=14, c=RND_C, edgecolor="none", label="reached once")
ax.scatter(b[many], kk[many], s=16, c=OPT_C, edgecolor="none", label="reached repeatedly")
ax.set_xscale("log"); ax.set_xlabel("runs reaching this topology")
ax.set_ylabel("complexity / median")
ax.legend(fontsize=7, loc="upper left")
axm = fig.add_subplot(gs[0,2], sharey=ax)
for i,(msk,c) in enumerate(((one,RND_C),(many,OPT_C))):
    if msk.any():
        axm.boxplot([kk[msk]], positions=[i], widths=.6, showfliers=False,
                    patch_artist=True, boxprops=dict(facecolor=c, edgecolor=INK, lw=.7),
                    medianprops=dict(color=INK, lw=1.1),
                    whiskerprops=dict(color=INK, lw=.7), capprops=dict(color=INK, lw=.7))
axm.set_xticks([0,1]); axm.set_xticklabels(["once","many"], fontsize=6.6)
axm.set_xlim(-.7,1.7); axm.tick_params(labelleft=False, left=False)
axm.spines["left"].set_visible(False); axm.set_xlabel("runs", fontsize=7)
fig.savefig(f"{OUT}/fig5.svg"); fig.savefig(f"{OUT}/fig5.png"); plt.show()
print(f"{len(runs)} converged runs -> {len(ks)} distinct topologies")
for m in MEASURES: print(f"  {NICE[m]:>8}: p = {pv[m]:.2e}")

## Figure 6

In [ ]:
def delta_of(cells, measure):
    cv = [c for c in cells if c.get("conv")] or cells
    dd = distinct(cv)
    if len(dd) < 2: return np.nan
    return cliffs_delta(np.array([K(c["xb"], measure) for c in dd]),
                        np.array([K(c["xb"], measure) for c in RND[:400]]))

panels = [("filter radius", [("1.5", S["designs"]["shear"])] +
           [(str(r), S["sweeps"][f"rmin={r}"]["shear"]) for r in RMIN_SWEEP[1:]]),
          ("mesh", [(str(n), S["sweeps"].get(f"N={n}",{}).get("shear", S["designs"]["shear"]
            if n == N else [])) for n in MESH_SWEEP]),
          ("base Poisson ratio", [(str(v), S["sweeps"].get(f"nu={v}",{}).get("shear",
            S["designs"]["shear"] if v == NU else [])) for v in NU_SWEEP])]
fig = plt.figure(figsize=(7.2,2.5))
gs = gridspec.GridSpec(1,3, wspace=.32, left=.07, right=.985, top=.86, bottom=.22)
for k,(nm,series) in enumerate(panels):
    ax = fig.add_subplot(gs[0,k])
    ax.set_title(f"({'abc'[k]})  {nm}", loc="left", fontsize=8.8, pad=6)
    labs = [s[0] for s in series]
    for m,c in zip(MEASURES,(OPT_C,ACC_C,ALT_C)):
        v = [delta_of(s[1], m) if s[1] else np.nan for s in series]
        ax.plot(range(len(series)), v, "o-", ms=3.4, lw=1.2, color=c, label=NICE[m])
    ax.set_xticks(range(len(series))); ax.set_xticklabels(labs, fontsize=7)
    ax.set_ylim(-1.1,.1); ax.axhline(0, color="#999999", lw=.8)
    if k == 0: ax.set_ylabel("Cliff's $\\delta$")
    if k == 2: ax.legend(fontsize=6.6, loc="lower right")
fig.savefig(f"{OUT}/fig6.svg"); fig.savefig(f"{OUT}/fig6.png"); plt.show()

## Supplementary figures

In [ ]:
def s2_radial(xb):
    x = np.asarray(xb, float); n = x.shape[0]
    F = np.fft.fft2(x)
    Sp = np.fft.fftshift(np.real(np.fft.ifft2(F*np.conj(F)))/x.size)
    c = n//2; yy, xx = np.mgrid[0:n,0:n]; dd = np.hypot(yy-c, xx-c)
    return np.arange(c+1, dtype=float), np.array(
        [Sp[(dd>=i-.5)&(dd<i+.5)].mean() for i in range(c+1)])

def feature_scale(r, pr, phi):
    tgt = phi*phi + (phi-phi*phi)/np.e
    for i in range(1, len(pr)):
        if pr[i] <= tgt:
            a, bb = pr[i-1], pr[i]
            return float(r[i-1] + (a-tgt)/(a-bb)) if a != bb else float(r[i])
    return float(r[-1])

def psd_radial(xb):
    x = np.asarray(xb, float); n = x.shape[0]
    P = np.abs(np.fft.fft2(x - x.mean()))**2
    P = np.fft.fftshift(P)/x.size
    c = n//2; yy, xx = np.mgrid[0:n,0:n]; dd = np.hypot(yy-c, xx-c)
    k = np.arange(1, c+1, dtype=float)
    return k, np.array([P[(dd>=i-.5)&(dd<i+.5)].mean() for i in range(1, c+1)])

grp_s3 = {"optima":[c["xb"] for c in OPT["shear"]+OPT["dirx"]][:120],
          "filtered random":[c["xb"] for c in RND[:120]],
          "correlation matched":[c["xb"] for c in MAT[:120]]}
fig = plt.figure(figsize=(7.0,2.7))
gs = gridspec.GridSpec(1,2, wspace=.32, left=.085, right=.985, top=.86, bottom=.19)
axP = fig.add_subplot(gs[0,0]); axP.set_title("(a)  power spectrum", loc="left",
                                              fontsize=8.8, pad=6)
axS = fig.add_subplot(gs[0,1]); axS.set_title("(b)  two-point correlation", loc="left",
                                              fontsize=8.8, pad=6)
store = {}
for (nm,cells),c,ls in zip(grp_s3.items(),(OPT_C,RND_C,ACC_C),("-","--",":")):
    k, Pm = psd_radial(cells[0])
    Pm = np.vstack([psd_radial(c2)[1] for c2 in cells]).mean(0)
    Pm = Pm/Pm.sum()
    r = s2_radial(cells[0])[0]
    Sm = np.vstack([s2_radial(c2)[1] for c2 in cells]).mean(0)
    phi = float(np.mean([np.mean(c2) for c2 in cells]))
    fs = feature_scale(r, Sm, phi)
    store[nm] = (Pm, Sm, phi, fs)
    axP.plot(k, Pm, ls, color=c, lw=1.5, label=nm)
    axS.plot(r, Sm, ls, color=c, lw=1.5,
             label=f"{nm} ($\phi$={phi:.3f}, $\ell$={fs:.2f})")
axP.set_xscale("log"); axP.set_yscale("log")
axP.set_xlabel("wavenumber k"); axP.set_ylabel("normalized power")
axP.legend(fontsize=6.6)
axS.set_xlabel("separation r (elements)"); axS.set_ylabel("$S_2(r)$")
axS.legend(fontsize=6.0)
fig.savefig(f"{OUT}/figS3.svg"); fig.savefig(f"{OUT}/figS3.png"); plt.show()
o = store["optima"]
for nm in ("filtered random","correlation matched"):
    v = store[nm]
    print(f"  {nm:22s}  max|dP| = {np.max(np.abs(o[0]-v[0])):.4f}   "
          f"max|dS2| = {np.max(np.abs(o[1]-v[1])):.4f}   "
          f"d(phi) = {abs(o[2]-v[2]):.4f}   d(scale) = {abs(o[3]-v[3]):.2f}")

In [ ]:
cellv = (OPT["shear"] or OPT["dirx"])[0]["xb"]
Hm = np.array([[K(np.roll(np.roll(cellv, dy, 0), dx, 1), "zlib")
                for dx in range(N)] for dy in range(N)])
spread = {}
for m in MEASURES:
    sp = []
    for c in (OPT["shear"] or OPT["dirx"])[:4]:
        v = np.array([K(np.roll(np.roll(c["xb"], dy, 0), dx, 1), m)
                      for dy in range(N) for dx in range(N)], float)
        sp.append((v.max()-v.min())/v.mean())
    spread[m] = sp

fig = plt.figure(figsize=(7.0,2.8))
gs = gridspec.GridSpec(1,2, width_ratios=[1,1.15], wspace=.34,
                       left=.075, right=.975, top=.84, bottom=.17)
ax = fig.add_subplot(gs[0,0]); ax.set_title("(a)  one cell, every origin", loc="left",
                                            fontsize=8.8, pad=7)
R = Hm/Hm.mean(); d = np.abs(R-1).max()
im = ax.imshow(R, cmap="RdBu_r", origin="lower", vmin=1-d, vmax=1+d)
ax.set_xlabel("shift in x"); ax.set_ylabel("shift in y")
cb = fig.colorbar(im, ax=ax, fraction=.046, pad=.03); cb.ax.tick_params(labelsize=7)
cb.set_label("complexity / mean", fontsize=7)
ax = fig.add_subplot(gs[0,1]); ax.set_title("(b)  spread across origins", loc="left",
                                            fontsize=8.8, pad=7)
for i,m in enumerate(MEASURES):
    v = 100*np.array(spread[m])
    ax.scatter(np.full(len(v),i)+np.random.default_rng(i).uniform(-.1,.1,len(v)),
               v, s=16, c=[OPT_C,ACC_C,ALT_C][i], edgecolor="none")
    ax.plot([i-.25,i+.25],[np.median(v)]*2, color=INK, lw=1.6)
ax.set_xticks(range(len(MEASURES))); ax.set_xticklabels([NICE[m] for m in MEASURES])
ax.set_ylabel("(max - min) / mean  (%)"); ax.set_xlim(-.6, len(MEASURES)-.4)
fig.savefig(f"{OUT}/figS4.svg"); fig.savefig(f"{OUT}/figS4.png"); plt.show()
for m in MEASURES: print(f"  {NICE[m]:>8}: median spread {100*np.median(spread[m]):.1f}%")

In [ ]:
def boot_rho(x, y, n=1000, seed=0):
    rs = np.random.default_rng(seed); out = []
    for _ in range(n):
        i = rs.integers(0, len(x), len(x))
        if len(np.unique(x[i])) > 2: out.append(spearmanr(x[i], y[i])[0])
    return (np.nan, np.nan) if not out else (float(np.percentile(out,2.5)),
                                             float(np.percentile(out,97.5)))
subsets = [("all", np.ones_like(b, bool)), ("largest dropped", b < b.max()),
           ("basin >= 2", many)]
fig = plt.figure(figsize=(7.0,2.9))
gs = gridspec.GridSpec(1,2, width_ratios=[1.3,1], wspace=.42,
                       left=.17, right=.975, top=.86, bottom=.28)
ax = fig.add_subplot(gs[0,0]); ax.set_title("(a)  basin size against complexity",
                                            loc="left", fontsize=8.8, pad=7)
cols = dict(zip([s[0] for s in subsets], (OPT_C, ALT_C, ACC_C)))
offs = dict(zip([s[0] for s in subsets], (.26, 0., -.26)))
for yi,m in enumerate(MEASURES[::-1]):
    for lab, msk in subsets:
        if msk.sum() < 4 or len(np.unique(b[msk])) < 3: continue
        r = spearmanr(b[msk], mk[m][msk])[0]; lo_, hi_ = boot_rho(b[msk], mk[m][msk])
        ax.plot([lo_,hi_],[yi+offs[lab]]*2, color=cols[lab], lw=2.3,
                solid_capstyle="round")
        ax.plot([r],[yi+offs[lab]], "o", color=cols[lab], ms=4.4,
                markeredgecolor="white", markeredgewidth=.6)
ax.axvline(0, color="#999999", lw=.9)
ax.set_yticks(range(len(MEASURES)))
ax.set_yticklabels([NICE[m] for m in MEASURES[::-1]])
ax.set_ylim(-.62, len(MEASURES)-.38)
ax.set_xlabel("rank correlation (95% bootstrap interval)")
for lab in cols: ax.plot([],[],"-",color=cols[lab],lw=2.3,label=lab)
ax.legend(fontsize=7, loc="upper center", ncol=3, bbox_to_anchor=(.5,-.22))
ax = fig.add_subplot(gs[0,1]); ax.set_title("(b)  degeneracy, by bin width", loc="left",
                                            fontsize=8.8, pad=7)
tols = [0.005, 0.01, 0.02, 0.05]; rs_ = []
for t in tols:
    g2 = {}
    for i,c in enumerate(pool): g2.setdefault(tkey(c["C"], t), []).append(i)
    s2 = np.array([len(v) for v in g2.values()], float)
    m2 = np.array([np.median(Kp[np.array(v)]) for v in g2.values()])
    rs_.append(spearmanr(s2, m2)[0])
ax.plot(tols, rs_, "o-", ms=4, lw=1.3, color=OPT_C)
ax.axhline(0, color="#999999", lw=.9); ax.set_xscale("log")
ax.set_xticks(tols); ax.set_xticklabels([f"{t:g}" for t in tols]); ax.minorticks_off()
ax.set_xlabel("binning tolerance"); ax.set_ylabel("rank correlation")
fig.savefig(f"{OUT}/figS5.svg"); fig.savefig(f"{OUT}/figS5.png"); plt.show()
print("  rho by tolerance:", {t: round(r,2) for t,r in zip(tols, rs_)})

## Tables

In [ ]:
print("Table S1\n")
print(f"{'objective':>12} {'estimator':>10} {'M':>5} {'n':>6} {'delta':>8} {'p':>11}")
for o,m,M,n,d,dm,p in rows:
    print(f"{o:>12} {NICE[m]:>10} {M:>5} {n:>6} {d:>8.2f} {p:>11.2e}")

print("\n\nTable S2\n")
print(f"{'condition':>18} {'objective':>12} {'runs':>6} {'conv':>6} "
      f"{'distinct':>9} {'+transl':>9}")
def line(cond, obj, cells):
    if not cells: return
    cv = [c for c in cells if c.get("conv")]
    nd = len(distinct(cv, key_d4)) if cv else 0
    nt = len(distinct(cv)) if cv else 0
    print(f"{cond:>18} {obj:>12} {len(cells):>6} {len(cv):>6} {nd:>9} "
          f"{(nt if cv else '-'):>9}")
for o in ("shear","dirx"): line(f"rmin={RMIN}", o, S["designs"][o])
for k, v in S["sweeps"].items():
    for o, cells in v.items(): line(k, o, cells)